Run this notebook to

- Create response epochs (300ms before to 1s after response, baselined at 100ms before to time of response)
- Drop epochs that fall within EEG parts annotated as bad
- Set behavioural meta-data
- Drop epochs that exceed peak-to-peak amplitude range of 200 µV 
- Sub-select task block data (ie not practice trials)
- Plot epoch timecourse for all channels
- Plot electrode Pz trace with stacked ERP plot sorted by reaction times
- Plot electrode Pz trace for correct vs error trials
- Calculate z-scored confidence reports within each partner condition and plot Pz trace for above vs below median confidence trials

The first part does this individually for each sessiona and each participant. The second part loops over both sessions and all participants.

## Import stuff and create directories

In [ ]:
import os
import numpy as np
import mne
from mne.preprocessing import (ICA, create_eog_epochs, create_ecg_epochs,corrmap)
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pandas as pd

million = 1000000.


%matplotlib qt

input_dir = 'EEGdata/cleaned'

In [ ]:
def mkdir(p):
    sp = re.split('/|\\\\', p)
    bp = ''
    for pp in sp:
        bp = os.path.join(bp, pp)
        if not os.path.exists(bp):
            os.mkdir(bp)
            print( '%s created.' % bp)

            
mkdir('TaskResponseEpochs')
mkdir('Figures/TaskResponseEpochs')
mkdir('Figures/TaskResponseEpochs/all channels')
mkdir('Figures/TaskResponseEpochs/correct vs error')
mkdir('Figures/TaskResponseEpochs/z-scored confidence median split')
mkdir('Figures/TaskResponseEpochs/ERP by reaction time')



In [ ]:
def plot_joint(erp, times, title='', width=12, height=8, invert=True, save=None):
    fig = erp.plot_joint(times=times,
                         show=False,
                         ts_args=dict(time_unit='s'),
                         topomap_args=dict(res=128, contours=4, time_unit='s'),
                         title=title)
    fig.set_figwidth(width)
    fig.set_figheight(height)
    axes = fig.get_axes()
    ax0 = axes[0]
    if invert:
        ax0.invert_yaxis()
    ch = ax0.get_children()
    for c in ch:
        if type(c) == plt.Annotation:
            c.remove()
        if type(c) == plt.Line2D:
            c.set_linewidth(2.5)
            c.set_alpha(.75)
    leg_ax = axes[-2]
    leg_ax.get_children()[0].set_sizes([50.])
    leg_ax.set_aspect('equal')
    if save is not None:
        fig.savefig(save)
    fig.show()

In [ ]:
subject = 1

In [ ]:
session = 1

## Load cleaned EEG data

In [ ]:
if session == 1:
    raw = mne.io.read_raw_fif('%s/s%i-raw.fif' % (input_dir, subject)).load_data() # session 1

if session == 2:
    raw = mne.io.read_raw_fif('%s/s%i_2-raw.fif' % (input_dir, subject)).load_data() # session 2

In [ ]:
if session == 1:
    annot_file = '%s/%i-annot.fif' % ('annotations', subject)

if session == 2:
    annot_file = '%s/%i_2-annot.fif' % ('annotations', subject)


annotations = mne.read_annotations(annot_file)
raw = raw.set_annotations(annotations)

## Create epochs

In [ ]:
# select response epochs and drop bad epochs by annotation

events = mne.find_events(raw, stim_channel='Status')

response_event_dict = {'response/left': 6,
                       'response/right': 7
                      }

response_epochs = mne.Epochs(raw, events, tmin=-0.3, tmax=1, event_id=response_event_dict, 
                             reject_by_annotation=True, preload=True, baseline=(-0.001, 0.001))



In [ ]:
drop_log = [v[0]  if (len(v) > 0) else 'ok' for v in response_epochs.drop_log]
drop_log = np.array(drop_log)
drop_log = drop_log[drop_log != 'IGNORED']

In [ ]:
if len(response_epochs) != len(drop_log[drop_log != 'BAD_']):
    raise Exception('Numbers of epochs do not match')

## Drop bad epochs from the metadata

In [ ]:
alldata = pd.read_csv("mergedData/allData.csv")
data = alldata[alldata['participant']== subject]


if session == 1:
    data = data[data['session']== 1] # session 1
if session == 2:
    data = data[data['session']== 2] # session 2

In [ ]:
data['droplog'] = drop_log
data = data[data['droplog'] == 'ok']
data.index = range(len(data))

In [ ]:
if len(response_epochs) != len(data):
    raise Exception('Number of epochs do not match.\n Found %i EEG events and %i csv events' % (len(epochs['response']), len(data)))


## Set metadata

In [ ]:
# attach metadata to epochs

metadata = pd.DataFrame(data=data)
response_epochs.metadata = metadata
response_epochs.metadata.head()

## Further drop epochs that exceed amplitude range

In [ ]:
# reject epochs with amplitude peak-to-peak amplitude range above 200 µV 

reject_criteria = dict(eeg=200e-6)
response_epochs.drop_bad(reject=reject_criteria)

## Sub-select task block data (i.e. not practice trials)

In [ ]:
task_response_epochs = response_epochs['block_count > 2']

In [ ]:
if session == 1:
    task_response_epochs.save('TaskResponseEpochs/%i_1-epo.fif' % (subject), overwrite=True) # session 1
if session == 2:
    task_response_epochs.save('TaskResponseEpochs/%i_2-epo.fif' % (subject), overwrite=True) # session 2

## Plot

In [ ]:
condition = task_response_epochs.metadata['condition'].unique()
condition = condition[0]

### Plot epoch all channels

In [ ]:
erp = task_response_epochs.copy().average()
plot_joint(erp, [-0.3, 0, 0.3, 0.6], title='Participant %i - Response-locked - %s' 
           % (subject, condition));

if session == 1:
    plt.savefig('Figures/TaskResponseEpochs/all channels/%i_1.png' % (subject)) # session 1

if session == 2:
    plt.savefig('Figures/TaskResponseEpochs/all channels/%i_2.png' % (subject)) # session 2

### Sort ERP by reaction time

In [ ]:
sort_order = np.argsort(task_response_epochs.metadata['decision_rt'])
task_response_epochs.plot_image(order=sort_order, picks='Pz')

if session == 1:
    plt.savefig('Figures/TaskResponseEpochs/ERP by reaction time/%i_1.png' % (subject)) # session 1
    
if session == 2:
    plt.savefig('Figures/TaskResponseEpochs/ERP by reaction time/%i_2.png' % (subject)) # session 2

### Correct vs error trials

In [ ]:
correct_epochs = task_response_epochs['participant_correct == True']
correct_evoked = correct_epochs.average()

incorrect_epochs = task_response_epochs['participant_correct == False']
incorrect_evoked = incorrect_epochs.average()


evokeds = dict(correct=correct_evoked,
               incorrect=incorrect_evoked)

mne.viz.plot_compare_evokeds(evokeds, picks=['Pz'], invert_y=False)
plt.title('Response-locked | P%i | %s' % (subject, condition))

if session == 1:
    plt.savefig('Figures/TaskResponseEpochs/correct vs error/%i_1.png' % (subject)) # session 1
    
if session == 2:
    plt.savefig('Figures/TaskResponseEpochs/correct vs error/%i_2.png' % (subject)) # session 2

### z-scored confidence median split

In [ ]:
zscore = lambda x: (x - x.mean()) / x.std()

task_response_epochs.metadata['confidence_z_by_partner'] = task_response_epochs.metadata['participant_confidence'].groupby(task_response_epochs.metadata['partner']).transform(zscore)

In [ ]:
median_confidence = task_response_epochs.metadata['confidence_z_by_partner'].median() # should be 0

low_conf_epochs = task_response_epochs['confidence_z_by_partner <= %i' % median_confidence]
high_conf_epochs = task_response_epochs['confidence_z_by_partner > %i' % median_confidence]

low_conf_evoked = low_conf_epochs.average()
high_conf_evoked = high_conf_epochs.average()


evokeds = dict(high_confidence=high_conf_evoked, low_confidence=low_conf_evoked)

mne.viz.plot_compare_evokeds(evokeds, picks=['Pz'], invert_y=False)
plt.title('Response-locked | P%i | %s' % (subject, condition))

if session == 1:
    plt.savefig('Figures/TaskResponseEpochs/z-scored confidence median split/%i_1.png' % (subject))

if session == 2:
    plt.savefig('Figures/TaskResponseEpochs/z-scored confidence median split/%i_2.png' % (subject))

## 
## 
## Loop everything

In [10]:
import os
import numpy as np
import mne
from mne.preprocessing import (ICA, create_eog_epochs, create_ecg_epochs,corrmap)
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pandas as pd

million = 1000000.


%matplotlib qt

input_dir = 'EEGdata/cleaned'



def mkdir(p):
    sp = re.split('/|\\\\', p)
    bp = ''
    for pp in sp:
        bp = os.path.join(bp, pp)
        if not os.path.exists(bp):
            os.mkdir(bp)
            print( '%s created.' % bp)

            
mkdir('TaskResponseEpochs')
mkdir('Figures/TaskResponseEpochs')
mkdir('Figures/TaskResponseEpochs/all channels')
mkdir('Figures/TaskResponseEpochs/correct vs error')
mkdir('Figures/TaskResponseEpochs/z-scored confidence median split')
mkdir('Figures/TaskResponseEpochs/ERP by reaction time')



def plot_joint(erp, times, title='', width=12, height=8, invert=True, save=None):
    fig = erp.plot_joint(times=times,
                         show=False,
                         ts_args=dict(time_unit='s'),
                         topomap_args=dict(res=128, contours=4, time_unit='s'),
                         title=title)
    fig.set_figwidth(width)
    fig.set_figheight(height)
    axes = fig.get_axes()
    ax0 = axes[0]
    if invert:
        ax0.invert_yaxis()
    ch = ax0.get_children()
    for c in ch:
        if type(c) == plt.Annotation:
            c.remove()
        if type(c) == plt.Line2D:
            c.set_linewidth(2.5)
            c.set_alpha(.75)
    leg_ax = axes[-2]
    leg_ax.get_children()[0].set_sizes([50.])
    leg_ax.set_aspect('equal')
    if save is not None:
        fig.savefig(save)
    fig.show()



sessions = [1, 2]

## CHECK PARTICIPANTS 5 and 8! (and 16 for dropping loads of epochs)

participant_numbers = [1, 2, 3, 4, 6, 7, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21]


for session in sessions:
    for subject in participant_numbers:
        print('>>>> Session %i' % session)
        print('>>>> Participant %i' % subject)
        
        if session == 1:
            raw = mne.io.read_raw_fif('%s/s%i-raw.fif' % (input_dir, subject)).load_data() # session 1
            annot_file = '%s/%i-annot.fif' % ('annotations', subject)

        if session == 2:
            raw = mne.io.read_raw_fif('%s/s%i_2-raw.fif' % (input_dir, subject)).load_data() # session 2
            annot_file = '%s/%i_2-annot.fif' % ('annotations', subject)


        annotations = mne.read_annotations(annot_file)
        raw = raw.set_annotations(annotations)


        events = mne.find_events(raw, stim_channel='Status')
        response_event_dict = {'response/left': 6,
                               'response/right': 7
                              }
        response_epochs = mne.Epochs(raw, events, tmin=-0.3, tmax=1, event_id=response_event_dict,
                                     reject_by_annotation=True, preload=True, baseline=(-0.1, 0))
        
        drop_log = [v[0]  if (len(v) > 0) else 'ok' for v in response_epochs.drop_log]
        drop_log = np.array(drop_log)
        drop_log = drop_log[drop_log != 'IGNORED']
        
        if len(response_epochs) != len(drop_log[drop_log != 'BAD_']):
            raise Exception('Numbers of epochs do not match')
        
        
        alldata = pd.read_csv("mergedData/allData.csv")
        data = alldata[alldata['participant']== subject]
        
        if session == 1:
            data = data[data['session']== 1] # session 1
        if session == 2:
            data = data[data['session']== 2] # session 2
            
        data['droplog'] = drop_log
        data = data[data['droplog'] == 'ok']
        data.index = range(len(data))
        
        if len(response_epochs) != len(data):
            raise Exception('Number of epochs do not match.\n Found %i EEG events and %i csv events' % (len(epochs['response']), len(data)))
        
        metadata = pd.DataFrame(data=data)
        response_epochs.metadata = metadata
        
        reject_criteria = dict(eeg=200e-6)
        response_epochs.drop_bad(reject=reject_criteria)
        
        task_response_epochs = response_epochs['block_count > 2']
        
        if session == 1:
            task_response_epochs.save('TaskResponseEpochs/%i_1-epo.fif' % (subject), overwrite=True) # session 1
        if session == 2:
            task_response_epochs.save('TaskResponseEpochs/%i_2-epo.fif' % (subject), overwrite=True) # session 2
            
        condition = task_response_epochs.metadata['condition'].unique()
        condition = condition[0]
        
        
        erp = task_response_epochs.copy().average()
        plot_joint(erp, [-0.3, 0, 0.3, 0.6], title='Participant %i - Response-locked - %s' % (subject, condition));
        if session == 1:
            plt.savefig('Figures/TaskResponseEpochs/all channels/%i_1.png' % (subject)) # session 1
        if session == 2:
            plt.savefig('Figures/TaskResponseEpochs/all channels/%i_2.png' % (subject)) # session 2
            
            
        sort_order = np.argsort(task_response_epochs.metadata['decision_rt'])
        task_response_epochs.plot_image(order=sort_order, picks='Pz')
        if session == 1:
            plt.savefig('Figures/TaskResponseEpochs/ERP by reaction time/%i_1.png' % (subject)) # session 1
        if session == 2:
            plt.savefig('Figures/TaskResponseEpochs/ERP by reaction time/%i_2.png' % (subject)) # session 2
            
            
        correct_epochs = task_response_epochs['participant_correct == True']
        correct_evoked = correct_epochs.average()
        incorrect_epochs = task_response_epochs['participant_correct == False']
        incorrect_evoked = incorrect_epochs.average()
        evokeds = dict(correct=correct_evoked,incorrect=incorrect_evoked)
        mne.viz.plot_compare_evokeds(evokeds, picks=['Pz'], invert_y=False)
        plt.title('Response-locked | P%i | %s' % (subject, condition))
        if session == 1:
            plt.savefig('Figures/TaskResponseEpochs/correct vs error/%i_1.png' % (subject)) # session 1
        if session == 2:
            plt.savefig('Figures/TaskResponseEpochs/correct vs error/%i_2.png' % (subject)) # session 2
        
        
        zscore = lambda x: (x - x.mean()) / x.std()
        task_response_epochs.metadata['confidence_z_by_partner'] = task_response_epochs.metadata['participant_confidence'].groupby(task_response_epochs.metadata['partner']).transform(zscore)
        
        median_confidence = task_response_epochs.metadata['confidence_z_by_partner'].median()
        low_conf_epochs = task_response_epochs['confidence_z_by_partner <= %i' % median_confidence]
        high_conf_epochs = task_response_epochs['confidence_z_by_partner > %i' % median_confidence]
        low_conf_evoked = low_conf_epochs.average()
        high_conf_evoked = high_conf_epochs.average()
        evokeds = dict(high_confidence=high_conf_evoked, low_confidence=low_conf_evoked)
        mne.viz.plot_compare_evokeds(evokeds, picks=['Pz'], invert_y=False)
        plt.title('Response-locked | P%i | %s' % (subject, condition))
        if session == 1:
            plt.savefig('Figures/TaskResponseEpochs/z-scored confidence median split/%i_1.png' % (subject))
        if session == 2:
            plt.savefig('Figures/TaskResponseEpochs/z-scored confidence median split/%i_2.png' % (subject))

Figures\TaskResponseEpochs created.
Figures\TaskResponseEpochs\all channels created.
Figures\TaskResponseEpochs\correct vs error created.
Figures\TaskResponseEpochs\z-scored confidence median split created.
Figures\TaskResponseEpochs\ERP by reaction time created.
>>>> Session 1
>>>> Participant 1
Opening raw data file EEGdata/cleaned/s1-raw.fif...
    Range : 0 ... 830749 =      0.000 ...  3322.996 secs
Ready.
Reading 0 ... 830749  =      0.000 ...  3322.996 secs...
2844 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
1 bad epochs dropped
Adding metadata with 50 columns
    Rejecting  epoch based on EEG : ['FT8']
    Rejecting  epoch based on EEG : ['AF4']
    Rejecting  epoch based on EEG : ['AF4']
    Rejecting  epoch based on EEG : ['FT8']
    Rejecting  epoch bas

c:\users\majaf\miniconda3\lib\site-packages\mne\viz\utils.py:1214: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  fig = plt.figure(figsize=figsize)


Not setting metadata
Not setting metadata
277 matching events found
No baseline correction applied
0 projection items activated
0 bad epochs dropped
>>>> Session 1
>>>> Participant 9
Opening raw data file EEGdata/cleaned/s9-raw.fif...
    Range : 0 ... 853999 =      0.000 ...  3415.996 secs
Ready.
Reading 0 ... 853999  =      0.000 ...  3415.996 secs...
Trigger channel has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2844 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
6 bad epochs dropped
Adding metadata with 50 columns
    Rejecting  epoch based on EEG : ['Fpz']
    Rejecting  epoch based on EEG : ['P9']
    Rejecting  epoch based on EEG : ['F4']
    Rejecting  epoc

Not setting metadata
Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
0 bad epochs dropped
>>>> Session 1
>>>> Participant 17
Opening raw data file EEGdata/cleaned/s17-raw.fif...
    Range : 0 ... 821749 =      0.000 ...  3286.996 secs
Ready.
Reading 0 ... 821749  =      0.000 ...  3286.996 secs...
2844 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
0 bad epochs dropped
Adding metadata with 50 columns
    Rejecting  epoch based on EEG : ['F8']
1 bad epochs dropped
Overwriting existing file.
No projector specified for this dataset. Please consider the method self.add_proj.
Not setting metadata
Not setting metadata
299 matching events found
No baseline correction applied
0 projection items activated
0 bad epoch

    Rejecting  epoch based on EEG : ['TP7', 'P9']
    Rejecting  epoch based on EEG : ['Fpz']
    Rejecting  epoch based on EEG : ['FC6']
    Rejecting  epoch based on EEG : ['T7']
    Rejecting  epoch based on EEG : ['Fpz']
    Rejecting  epoch based on EEG : ['T7']
    Rejecting  epoch based on EEG : ['P9']
    Rejecting  epoch based on EEG : ['P7', 'P9']
    Rejecting  epoch based on EEG : ['FT8']
    Rejecting  epoch based on EEG : ['FT8']
    Rejecting  epoch based on EEG : ['FT8']
    Rejecting  epoch based on EEG : ['FCz']
27 bad epochs dropped
Overwriting existing file.
No projector specified for this dataset. Please consider the method self.add_proj.
Not setting metadata
Not setting metadata
277 matching events found
No baseline correction applied
0 projection items activated
0 bad epochs dropped
>>>> Session 2
>>>> Participant 3
Opening raw data file EEGdata/cleaned/s3_2-raw.fif...
    Range : 0 ... 625499 =      0.000 ...  2501.996 secs
Ready.
Reading 0 ... 625499  =      0.

0 projection items activated
0 bad epochs dropped
>>>> Session 2
>>>> Participant 11
Opening raw data file EEGdata/cleaned/s11_2-raw.fif...
    Range : 0 ... 690249 =      0.000 ...  2760.996 secs
Ready.
Reading 0 ... 690249  =      0.000 ...  2760.996 secs...
Trigger channel has a non-zero initial value of 65536 (consider using initial_event=True to detect this event)
Removing orphaned offset at the beginning of the file.
2239 events found
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11 15]
Not setting metadata
Not setting metadata
405 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 405 events and 326 original time points ...
1 bad epochs dropped
Adding metadata with 50 columns
    Rejecting  epoch based on EEG : ['Fp1', 'F5']
1 bad epochs dropped
Overwriting existing file.
No projector specified for this dataset. Please consider the method self.add_proj.
Not setting metadata
Not setting metadata
299 matching events found
No b

    Rejecting  epoch based on EEG : ['F5', 'F7', 'FT7', 'FC5', 'FC3', 'C3', 'C5', 'T7', 'TP7']
    Rejecting  epoch based on EEG : ['C1']
    Rejecting  epoch based on EEG : ['F5', 'F7']
    Rejecting  epoch based on EEG : ['AF7', 'F3', 'F5', 'F7', 'FT7', 'FC5', 'FC3', 'C3', 'C5', 'T7', 'TP7', 'CP5', 'CP3']
    Rejecting  epoch based on EEG : ['AF7', 'F3', 'F5', 'F7', 'FC5', 'TP8', 'P8', 'P10', 'PO8']
    Rejecting  epoch based on EEG : ['F7']
    Rejecting  epoch based on EEG : ['FT8']
    Rejecting  epoch based on EEG : ['AF7', 'F3', 'F5', 'F7', 'FT7', 'FC5', 'FC3', 'FC1', 'C3', 'C5', 'T7', 'TP7', 'CP5', 'CP3', 'P5', 'Iz']
    Rejecting  epoch based on EEG : ['F7']
    Rejecting  epoch based on EEG : ['FC1']
    Rejecting  epoch based on EEG : ['F5', 'F7', 'FT7', 'FC5', 'FC3', 'C3', 'T7', 'TP7']
    Rejecting  epoch based on EEG : ['AF7', 'F3', 'F5', 'F7', 'FT7', 'FC5', 'FC3', 'FC1', 'C3', 'C5', 'T7', 'TP7', 'CP5', 'CP3']
    Rejecting  epoch based on EEG : ['F3', 'F5', 'F7', 'FT7', 

    Rejecting  epoch based on EEG : ['C6']
    Rejecting  epoch based on EEG : ['FT7', 'FC5', 'C5', 'CP5']
    Rejecting  epoch based on EEG : ['FC6', 'C6']
    Rejecting  epoch based on EEG : ['FC6']
    Rejecting  epoch based on EEG : ['FC6', 'C6']
32 bad epochs dropped
Overwriting existing file.
No projector specified for this dataset. Please consider the method self.add_proj.
Not setting metadata
Not setting metadata
267 matching events found
No baseline correction applied
0 projection items activated
0 bad epochs dropped


## TO DO:

- sanity check reaction time in behavioural datra vs diff in trigger times (stimulus - response) (plot the correlation (scatterplot) of those and there should not be any weird data)
- ERPs by confidence quartiles
- ERPs by overconfident vs underconfident partner
- check 5_2 annotations
- check 8_2 epochs not matching
- check 16_2 loads of epochs dropped
- make a list with remaining epochs per participant (just add len(epochs) to some list) to check if too many epochs are dropped for anyone
- see if you can do the P3 sorted by reaction time plot with smoothing across epochs (rather than just temporal smoothing within epochs) to maybe bring out the diagonal red line more with P3 starting later for longer reaction times
- do analyses as discussed in meeting with Jasmine and Nick


## Ideas from lab meeting
- check how meta-d realtes to neural signature of confidence (do people with better metacognitive abilities have clearer components?)
- translation/mapping from privat to public confidence --> maybe when trying to decode confidence by timepoints, true private confidence can be decoded earlier and public confidence is better decoded at a later stage (because it first needs to be mapped from private to public)
- the early late thing might work better in a task where you are told which partner you work with on a trial by trial basis. ie you make your own confidence judgement internally, then you see which partner it is (eg indicated by a color), and then you need to map accordingly on the spot to the appropriate public confidence
- train classifier on strategic and then contrast it with observational and see which parts help to decode confidence in observational and which parts are relevant for strategic only and those may be the ones that are relevant for the mapping
- how much noise in reporting confidence comes from actual p(correct) to neural signature and how much comes from neural signature to behavioural manifestation as a confidence report on a scale?